In [ ]:
import random
from pathlib import Path
from statistics import mean
import numpy as np
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import nn
import torchvision.transforms as transforms

import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import timm
from pytorch_datasets import ISIC2018, BreakHis, Papsmear
from pytorch_datasets import augmentations

from easyfsl.samplers import TaskSampler
# from easyfsl.methods.utils import evaluate
from easyfsl.utils import evaluate
from torch.utils.data import DataLoader
from easyfsl.datasets import MiniImageNet
from easyfsl.datasets.default_configs import default_transform

from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import math
import copy

from torch.utils.tensorboard import SummaryWriter

### Initial setup

In [ ]:
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)
random.seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda"
IMAGENET_NORMALIZATION = {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}

In [ ]:
cutout_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.RandomHorizontalFlip(),
    augmentations.Cutout(n_holes=1, length=64),
    transforms.ToTensor(),
    transforms.Normalize(**IMAGENET_NORMALIZATION),
])
simple_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

train_transform = default_transform(image_size=224, training=True)
test_transform  = default_transform(image_size=224, training=False)

In [ ]:
# Training Parameters
n_way = 2
n_shot = 3
n_query = 10

n_episodes = 40000
n_tasks_per_epoch = 500
n_validation_tasks = 200

n_workers = 6

cd = False
cd_base_dataset = 'IN'
cd_target_dataset = 'ISIC'

# Selecting augmentation
# augments = [None, 'cutout', 'mixup', 'cutmix']
# augmentation = augments[0]

# if augmentation == 'cutout':
#     transform = cutout_transform
# else:
#     transform = None


# Selecting a dataset
datasets = {0: 'ISIC2018', 1: 'BreakHis', 2: 'Papsmear'}
dataset = datasets[1]  # Dataset ID

if dataset == 'ISIC2018':
    root = 'C:/Datasets/ISIC2018'
    train_set = ISIC2018(root=root, split='train',
                         training=True,)
    val_set = ISIC2018(root=root, split='validation',
                       training=False,)
elif dataset == 'BreakHis':
    root = 'C:/Datasets/BreaKHis_v1'
    magnification = 40

    train_set = BreakHis(root=root, magnification=magnification,
                         split='train', training=True,)
    val_set = BreakHis(root=root, magnification=magnification,
                       split='validation', training=False,)
elif dataset == 'Papsmear':
    train_root = r"C:\Datasets\smear2005\New database pictures\train"
    test_root = r"C:\Datasets\smear2005\New database pictures\test"

    train_set = Papsmear(root=train_root, training=True,)
    val_set = Papsmear(root=test_root, training=False,)

# if cd:
#     if cd_base_dataset == "ISIC":
#         root = 'C:/Datasets/ISIC2018'
#         train_set = ISIC2018(root=root, split='train',
#                              training=True, transforms=transform)
#     elif cd_base_dataset == 'IN':
#         train_specs_file = r"E:\Datasets\ImageNet1k\mini_imagenet\train.csv"
#         train_set = MiniImageNet(root=r"C:\Datasets\miniIN\train",
#                                  specs_file=train_specs_file,
#                                  training=True,
#                                  transform=train_transform,)

#     if cd_target_dataset == 'ISIC':
#         root = 'C:/Datasets/ISIC2018'
#         val_set = ISIC2018(root=root, split='validation',
#                            training=False, transforms=transform)
#     elif cd_target_dataset == 'Papsmear':
#         test_root = r"C:\Datasets\smear2005\New database pictures\test"
#         val_set = Papsmear(root=test_root, training=False, transform=transform)
#     elif cd_target_dataset == 'BreakHis':
#         root = 'C:/Datasets/BreaKHis_v1'
#         magnification = 100
#         val_set = BreakHis(root=root, magnification=magnification,
#                            split='validation', training=False, transforms=transform)

train_sampler = TaskSampler(
    train_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_tasks_per_epoch
)
test_sampler = TaskSampler(
    val_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_validation_tasks
)

# Finally, the DataLoader. We customize the collate_fn so that batches are delivered
# in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
train_loader = DataLoader(
    train_set,
    batch_sampler=train_sampler,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=train_sampler.episodic_collate_fn,
    # persistent_workers=True,
)
val_loader = DataLoader(
    val_set,
    batch_sampler=test_sampler,
    num_workers=n_workers,
    pin_memory=True,
    # persistent_workers=True,
    collate_fn=test_sampler.episodic_collate_fn,
)


In [ ]:
# labels = np.array(train_set.get_labels())

In [ ]:
# from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# compute_class_weight(class_weight='balanced', classes=[1,4,5,6,7] , y= labels)

In [ ]:
class_weights_isic  = torch.tensor([0.35164057, 2.11837376, 2.14535942, 1, 1,  4.58706226, 1]).cuda()
class_weights_break = torch.tensor([1, 1.30538462, 1, 1, 0.37585825, 1.99647059, 1.52882883, 2.39014085]).cuda()
class_weights_papsmear = torch.tensor([1.125, 0.9271978, 1.15582192, 0.85659898]).cuda()

## Training

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# timm.list_models()

### Pre-train for transfer learning

In [ ]:
# batch_size = 128

# train_size = int(0.95 * len(train_set))
# val_size = len(train_set) - train_size
# trainset, valset= torch.utils.data.random_split(train_set, [train_size, val_size])
# valset = copy.deepcopy(valset)

# valset.dataset.transforms = val_set.transforms


# train_loader = DataLoader(
#     trainset,
#     batch_size=batch_size,
#     shuffle=True,
#     num_workers=n_workers,
#     pin_memory=True,
#     persistent_workers=True,                                     
# )
# val_loader = DataLoader(
#     valset, 
#     batch_size=batch_size, 
#     shuffle=False, 
#     num_workers=n_workers, 
#     pin_memory=True,
#     persistent_workers=True,
# )

In [ ]:
# # lrs = [0.1, 0.03, 0.01]
# lrs = [0.01]
# # model_name = 'resnet50'
# model_name = 'vit_small_patch16_224'

# device = DEVICE

# if dataset == "ISIC2018":
#     weights = class_weights_isic
#     n_classes = 7
# elif dataset == 'BreakHis':
#     weights = class_weights_break
#     n_classes = 8
# elif dataset == 'Papsmear':
#     weights = class_weights_papsmear
#     n_classes = 4

# for lr in lrs:
#     model = timm.create_model(model_name, pretrained=True, num_classes=n_classes).cuda()

#     warmup_steps = 100
#     steps = 500
#     num_epochs = math.ceil(steps / len(train_loader))
    
#     loss_fn = nn.CrossEntropyLoss(weight=weights)
#     # optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-5)
#     # optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=0.1)
#     optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

#     # Define loss function and optimizer
#     warmup_scheduler = LinearLR(
#         optimizer,
#         start_factor=1/warmup_steps,
#         end_factor=1, 
#         total_iters=warmup_steps
#     )
#     cosine_scheduler = CosineAnnealingLR(
#         optimizer,
#         T_max=steps-warmup_steps,
#         eta_min=0
#     )

#     scheduler = SequentialLR(
#         optimizer, 
#         [warmup_scheduler, cosine_scheduler],
#         milestones=[warmup_steps]
#     )

#     writer = SummaryWriter(comment=f'{dataset}_{model_name}_bs{batch_size}_lr{lr}nogc')
#     # writer.add_hparams(hparam_dict=hparams, metric_dict={'acc': 0})

#     #@title training steps
#     best_model = None
#     best_val_loss = float('inf')

#     use_amp = True
#     scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

#     pbar = tqdm(total=steps, smoothing = 0)
#     step = 1

#     tb_write = True

#     for epoch in range(num_epochs):
#         for (inputs, labels) in train_loader:
#             model.train()

#             inputs, labels = inputs.to(device), labels.to(device)

#             optimizer.zero_grad()
#             with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
#                 outputs = model(inputs)
#                 loss    = loss_fn(outputs, labels)
            
#             scaler.scale(loss).backward()
#             scaler.unscale_(optimizer)

#             nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
#             scaler.step(optimizer)
#             scaler.update()
#             # Update the learning rate scheduler
#             scheduler.step()
            
#             if tb_write:
#                 writer.add_scalar("Loss/train", loss.item(), step)
#                 writer.add_scalar("lr", scheduler.get_last_lr()[0], step)

#             if step % 10 == 0:
#                 # Evaluation phase
#                 model.eval()
#                 val_loss = 0.0
#                 val_correct = 0
                
#                 with torch.no_grad():
#                     for inputs, labels in val_loader:
#                         inputs, labels = inputs.to(device), labels.to(device)

#                         with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
#                             outputs = model(inputs)
#                             loss = loss_fn(outputs, labels)

#                         val_loss += loss.item() * inputs.size(0)
#                         val_correct += torch.sum(torch.argmax(outputs, dim=1) == labels)

#                     val_loss /= len(val_loader.dataset)
#                     val_accuracy = val_correct.float() / len(val_loader.dataset)
                
#                     if tb_write:
#                         writer.add_scalar("Loss/test", val_loss, step)
#                         writer.add_scalar("acc", val_accuracy, step)
            
#             pbar.update(1)
#             if(step == steps):
#                 break
#             step += 1
#             # if val_loss < best_val_loss:
#             #     best_val_loss = val_loss
#             #     best_model = model.state_dict()
#             #     print('!!! New best model found !!!')
#             #     # torch.save(best_model, 'best_model.pth')

#     torch.save(model, f'E:/Models/{dataset}_{model_name}_bs{batch_size}_lr{lr}.pth')

### Model selection

In [ ]:
# from easyfsl.methods import PrototypicalNetworks, FewShotClassifier, RelationNetworks, MatchingNetworks

# # is_custom = True
# is_custom = False

# if is_custom:
#     # model = torch.load(r"E:\Models\miniIN_convit_small_steps400_bs64_lr0.03_useSEFalse.pth")
#     # model = torch.load(r"E:\Models\convit_small_bs128_lr0.1_useSETrue2.pth")
#     # model = torch.load(r"E:\Models\ImageNet1k\coatnet_0_rw_224_bs128_accum4_steps500_warm50.0_lr0.03_useSETrue_Freeze.pth")
#     model = torch.load(r"E:\Models\protonet_checkpoints\vit_small_patch16_224.pt")
#     # model = torch.load(r"E:\Models\cifar100\vit_small_patch16_224_bs128_accum1_steps1500_warm150.0_lr0.03_useSETrue_ratio0.5.pth")
#     # model = torch.load(r"E:\Models\ImageNet1k\vit_small_patch16_224_bs128_accum4_steps500_warm50.0_lr0.03_useSETrue_Freeze.pth")
    
#     for param in model.parameters():
#         param.requires_grad = True
#     # model.head.flatten[0] = nn.Identity() # coatSE
#     # model.head.fc = nn.Identity() # coat 
#     model.head = nn.Identity() # vit
#     # model.head[1] = nn.Identity() # vit SE
# else:
#     # model_name = 'convit_small'
#     model_name = 'vit_small_patch16_224'
#     # model_name = 'coatnet_0_rw_224'
#     # model_name = 'resnet50'
#     # model_name = 'mobilevitv2_050'

#     model = timm.create_model(model_name, pretrained=True, in_chans=3)

#     if model_name == 'coatnet_0_rw_224':
#         model.head.fc = nn.Identity() # coat
#     if model_name in ['vit_small_patch16_224', 'vit_base_patch16_224']:
#         model.head = nn.Identity() # vit


# # model.head.flatten.fc = nn.Identity() # swin

# # model.fc = nn.Identity() # resnet50
# # model.fc = nn.Identity() # mvit

# model = model.cuda()
# few_shot_classifier = PrototypicalNetworks(model).to(DEVICE)
# # few_shot_classifier = MatchingNetworks(model).to(DEVICE)

In [ ]:
from easyfsl.methods import PrototypicalNetworks, FewShotClassifier, RelationNetworks, MatchingNetworks
from easyfsl.modules import resnet12

model_name = 'vit_small_patch16_224'
# model_name = 'vit_tiny_patch16_224'
# model_name = 'vit_tiny_r_s16_p8_224'
# model_name = 'swin_base_patch4_window7_224'
# model_name = 'vgg16'
# model_name = 'resnet50'
# model_name = 'mobilevitv2_050'

# model = timm.create_model(model_name, pretrained=True, in_chans=3)
# model = torch.load('/content/drive/MyDrive/sevit_tiny.pth')

# model.head = nn.Identity() # vit
# vit.head.fc = nn.Identity() # swin
# vit.head.fc = nn.Identity() # vgg16
# model.fc = nn.Identity() # resnet50
# vit.fc = nn.Identity() # mvit
# vit = torch.load('/content/drive/MyDrive/hybrid_vit_cpt.pth')
# model.head[1] = nn.Identity()

# model = model.cuda()
# few_shot_classifier = MatchingNetworks(model, feature_dimension=384).to(DEVICE)
# few_shot_classifier = PrototypicalNetworks(model).to(DEVICE)
# few_shot_classifier = RelationNetworks(model).to(DEVICE)

In [ ]:
count_parameters(few_shot_classifier)

In [ ]:
lr = 0.01
batch_size = 128

In [ ]:
def load_model(model_name, mag):
    model = torch.load(f'E:/Models/{dataset}_mag{mag}_{model_name}_bs{batch_size}_lr{lr}.pth')
    
    if model_name == 'vit_small_patch16_224':
        model.head = nn.Identity() # vit
    else:
        model.fc = nn.Identity() # resnet50

# vit.head.fc = nn.Identity() # swin
# vit.head.fc = nn.Identity() # vgg16
# model.fc = nn.Identity() # resnet50
# vit.fc = nn.Identity() # mvit
# vit = torch.load('/content/drive/MyDrive/hybrid_vit_cpt.pth')
# model.head[1] = nn.Identity()

# model = model.cuda()
few_shot_classifier = MatchingNetworks(model, feature_dimension=384).to(DEVICE)
    return PrototypicalNetworks(model).to(DEVICE)

## Meta-train

In [ ]:
from torch.optim import SGD, Optimizer
from torch.optim.lr_scheduler import MultiStepLR, LinearLR, SequentialLR

# if augmentation == 'mixup':
#     LOSS_FUNCTION = augmentations.mixup_criterion()
# else:
# LOSS_FUNCTION = nn.CrossEntropyLoss()
LOSS_FUNCTION = nn.NLLLoss()

n_epochs = 10
scheduler_milestones = [120, 160]
warmup_epochs = 3
scheduler_gamma = 0.1
learning_rate = 1e-5
tb_logs_dir = Path(".")

train_optimizer = SGD(
    few_shot_classifier.parameters(), lr=learning_rate, momentum=0.9
)

# warmup_scheduler = LinearLR(
#     train_optimizer,
#     start_factor=0.02,
#     end_factor=1, 
#     total_iters=warmup_epochs
# )
# cosine_scheduler = CosineAnnealingLR(
#     train_optimizer,
#     T_max=n_epochs-warmup_epochs,
#     eta_min=1e-6
# )

# seq_scheduler = SequentialLR(
#     train_optimizer, 
#     [warmup_scheduler, cosine_scheduler],
#     milestones=[warmup_epochs]
# )


train_scheduler = MultiStepLR(
    train_optimizer,
    milestones=scheduler_milestones,
    gamma=scheduler_gamma,
)

# train_scheduler = seq_scheduler

In [ ]:
def training_epoch(
    model: FewShotClassifier, data_loader: DataLoader, optimizer: Optimizer, criterion, augmentation: str, alpha=0.4
):
    all_loss = []
    model.train()
    with tqdm(
        enumerate(data_loader), total=len(data_loader), desc="Training"
    ) as tqdm_train:
        for episode_index, (
            support_images,
            support_labels,
            query_images,
            query_labels,
            _,
        ) in tqdm_train:
            optimizer.zero_grad() 
            model.process_support_set(
                support_images.to(DEVICE), support_labels.to(DEVICE)
            )
            classification_scores = model(query_images.to(DEVICE))
            
            loss = criterion(classification_scores, query_labels.to(DEVICE))
            loss.backward()
            optimizer.step()

            all_loss.append(loss.item())

            tqdm_train.set_postfix(loss=mean(all_loss))

    return mean(all_loss)

In [ ]:
use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def training_epoch_tensor(
    model: FewShotClassifier, data_loader: DataLoader, optimizer: Optimizer, criterion, alpha=0.4
):
    all_loss = []
    model.train()
    with tqdm(
        enumerate(data_loader), total=len(data_loader), desc="Training"
    ) as tqdm_train:
        for episode_index, (
            support_images,
            support_labels,
            query_images,
            query_labels,
            _,
        ) in tqdm_train:
            optimizer.zero_grad() 
            
            model.process_support_set(
                support_images.to(DEVICE), support_labels.to(DEVICE)
            )
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                classification_scores = model(query_images.to(DEVICE))
                loss = criterion(classification_scores, query_labels.to(DEVICE))

            scaler.scale(loss).backward()
            scaler.step(optimizer)

            scaler.update()
            all_loss.append(loss.item())

            tqdm_train.set_postfix(loss=mean(all_loss))

    return mean(all_loss)

### Train

In [ ]:
best_state = few_shot_classifier.state_dict()
best_validation_accuracy = 0.0
for epoch in range(n_epochs):
    print(f"Epoch {epoch}")
    average_loss = training_epoch_tensor(few_shot_classifier, train_loader, train_optimizer, LOSS_FUNCTION)
    validation_accuracy = evaluate(
        few_shot_classifier, val_loader, device=DEVICE, tqdm_prefix="Validation"
    )

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_state = few_shot_classifier.state_dict()
        print(f"Ding ding ding! We found a new best model! Accuracy {best_validation_accuracy}")

    # tb_writer.add_scalar("Train/loss", average_loss, epoch)
    # tb_writer.add_scalar("Val/acc", validation_accuracy, epoch)

    # Warn the scheduler that we did an epoch
    # so it knows when to decrease the learning rate
    train_scheduler.step()

In [ ]:
# best_state = few_shot_classifier.state_dict()
# best_validation_accuracy = 0.0

# for epoch in range(n_epochs):
#     print(f"Epoch {epoch}")
#     average_loss = training_epoch_tensor(few_shot_classifier, train_loader, train_optimizer, LOSS_FUNCTION, augmentation)
#     validation_accuracy = evaluate(
#         few_shot_classifier, val_loader, device=DEVICE, tqdm_prefix="Validation"
#     )

#     if validation_accuracy > best_validation_accuracy:
#         best_validation_accuracy = validation_accuracy
#         best_state = few_shot_classifier.state_dict()
#         print("Ding ding ding! We found a new best model!")

#     # tb_writer.add_scalar("Train/loss", average_loss, epoch)
#     # tb_writer.add_scalar("Val/acc", validation_accuracy, epoch)

#     # Warn the scheduler that we did an epoch
#     # so it knows when to decrease the learning rate
#     train_scheduler.step()

In [ ]:
latest = few_shot_classifier.state_dict()

In [ ]:
few_shot_classifier.load_state_dict(latest)

## Evaluation

In [ ]:
mags = [40, 200, 400]
models = ['resnet50', 'vit_small_patch16_224']
#Evaluation Loop
ways   = [2, 3]
shots  = [3, 5, 10]
n_query = 10

n_validation_tasks  = 400

n_workers = 4

for mag in mags:
  root = 'C:/Datasets/BreaKHis_v1'
  magnification = mag
  
  val_set = BreakHis(root=root, magnification=magnification,
                      split='validation', training=False,)
  for model_name in models:
    few_shot_classifier = load_model(model_name, mag)
    for way in ways:
      for shot in shots:
        test_sampler = TaskSampler(
            val_set, n_way=way, n_shot=shot, n_query=n_query, n_tasks=n_validation_tasks
        )

        # Finally, the DataLoader. We customize the collate_fn so that batches are delivered
        # in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
        test_loader = DataLoader(
            val_set,
            batch_sampler=test_sampler,
            num_workers=6,
            pin_memory=True,
            collate_fn=test_sampler.episodic_collate_fn,
        )

        accuracy = evaluate(few_shot_classifier, test_loader, device=DEVICE)
        print(f"{model_name}_mag{mag}_{way}_{shot}_Average accuracy : {(100 * accuracy):.2f} %")

In [ ]:
# for dataset in datasets.values():
#     cd_target_dataset = dataset

#     if cd_target_dataset == 'ISIC2018':
#         root = 'C:/Datasets/ISIC2018'
#         val_set = ISIC2018(root=root, split='validation',
#                             training=False, transforms=transform)
#     elif cd_target_dataset == 'Papsmear':
#         test_root = r"C:\Datasets\smear2005\New database pictures\test"
#         val_set = Papsmear(root=test_root, training=False, transform=transform)
#     elif cd_target_dataset == 'BreakHis':
#         root = 'C:/Datasets/BreaKHis_v1'
#         magnification = 100
#         val_set = BreakHis(root=root, magnification=magnification,
#                             split='validation', training=False, transforms=transform)
#     #Evaluation Loop
#     ways   = [2, 3]
#     shots  = [3, 5, 10]
#     n_query = 10

#     n_validation_tasks  = 300

#     n_workers = 4

#     for way in ways:
#         for shot in shots:
#             test_sampler = TaskSampler(
#                 val_set, n_way=way, n_shot=shot, n_query=n_query, n_tasks=n_validation_tasks
#             )

#             # Finally, the DataLoader. We customize the collate_fn so that batches are delivered
#             # in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
#             test_loader = DataLoader(
#                 val_set,
#                 batch_sampler=test_sampler,
#                 num_workers=n_workers,
#                 pin_memory=True,
#                 collate_fn=test_sampler.episodic_collate_fn,
#             )

#             accuracy = evaluate(few_shot_classifier, test_loader, device=DEVICE)
#             print(f"Average accuracy : {(100 * accuracy):.2f} %")

In [ ]:
# accuracy = evaluate(few_shot_classifier, val_loader, device=DEVICE)
# print(f"Average accuracy : {(100 * accuracy):.2f} %")

In [ ]:
# torch.save(few_shot_classifier, f'E:/Models/protonet_checkpoints/{model_name}.pt')